# Notebook 04 — Feature Engineering

## Objective

This notebook transforms the cleaned banking data into a customer-level analytical dataset.

The feature engineering process includes:

- customer profile features;
- account portfolio features;
- financial features;
- product ownership features;
- temporal and compliance features.

The final dataset will respect the following granularity:

> One row represents one unique customer.

In [14]:
# ==========================================================
# 4.1 IMPORTS AND CONFIGURATION
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

print("=" * 70)
print("NOTEBOOK 04 — FEATURE ENGINEERING")
print("=" * 70)

NOTEBOOK 04 — FEATURE ENGINEERING


In [15]:
# ==========================================================
# 4.2 PROJECT PATHS
# ==========================================================

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_DATA_DIR = PROJECT_ROOT / "data" / "features"
REPORT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

FEATURE_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = (
    PROCESSED_DATA_DIR
    / "customer_clean_dataset.csv"
)

OUTPUT_FILE = (
    FEATURE_DATA_DIR
    / "customer_feature_dataset.csv"
)

print("Project root :", PROJECT_ROOT)
print("Input file   :", INPUT_FILE)
print("Input exists :", INPUT_FILE.exists())
print("Output file  :", OUTPUT_FILE)

Project root : c:\Users\Sarra\OneDrive\Desktop\PFE_CHURN_ESB
Input file   : c:\Users\Sarra\OneDrive\Desktop\PFE_CHURN_ESB\data\processed\customer_clean_dataset.csv
Input exists : True
Output file  : c:\Users\Sarra\OneDrive\Desktop\PFE_CHURN_ESB\data\features\customer_feature_dataset.csv


In [16]:
# ==========================================================
# 4.3 LOAD CLEAN DATASET
# ==========================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        "Clean dataset not found.\n"
        f"Expected location: {INPUT_FILE}\n"
        "Re-run the corrected export cells in Notebook 03."
    )

df = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig",
    low_memory=False
)

print("=" * 70)
print("CLEAN DATASET LOADED")
print("=" * 70)
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

display(df.head())

CLEAN DATASET LOADED
Rows    : 490,244
Columns : 34


,CUSTOMER_NO,ACCOUNT_NO,NATIONALITY,RESIDENCE,MARITAL_STATUS,CUST_OPENING_DATE,DATE_OF_BIRTH,NATURE_CLIENT,BRANCH,SCORE_KYC,COMPLETED_FILE,LAST_REVIEW_DATE,NEXT__REVIEW_DATE,ACCOUNT_STATUS,ACCT_OPENING_DATE,ACCOUNT_CATEGORY,ACCOUNT_TYPE_DESC,CURRENCY,ACCT_CLOSE_DATE,CLOSURE_REASON,ACCT_BALANCE,INDUSTRY,SALARY,PRODUCT_GROUP,PRODUCT_LINE,PRODUCT,ACCOUNTNATURE,STARTDATE,MATURITYDATE,AMOUNT,FIXEDRATE,PRODUCT_STATUS,PARTYCLASS,LOB
0,113772024,2.012013e+09,TN,TN,M,2004-09-30,1969-05-29,PPH,5820,LR,YES,2025-09-05,2029-09-05,Closed,2019-08-27,3023.0,Crédit acquisition logement TEGF6,TND,2026-01-28,NOT_APPLICABLE,-10714.347,9000,2725.739,RT.CRD.IMMOBILERS,LENDING,RT.RT.CRD.IMMOBILERS.527,Crédit acquisition logement TEGF6,1251227.0,1290627.0,10954600.0,4.5,CURRENT,Retail,4
1,113772022,2.012119e+09,TN,TN,M,2004-09-30,1960-09-23,PPH,5820,LR,YES,2025-09-05,2029-09-05,Closed,2026-01-05,3611.0,DEPOTS A TERME,TND,2025-06-30,NOT_APPLICABLE,0.000,9000,3300.537,ATB.PLACEMENT.NEG,DEPOSITS,ATB.CAT.NEG.SIM,DEPOTS A TERME,20260102.0,NaN,NaN,NaN,UNAUTH,Retail,4
2,113772024,2.011691e+09,TN,TN,M,2004-09-30,1969-05-29,PPH,5820,LR,YES,2025-09-05,2029-09-05,Closed,2023-06-12,3017.0,Crédit rénovation,TND,2026-01-28,NOT_APPLICABLE,-113033.101,9000,2725.739,RT.CRD.IMMOBILERS,LENDING,RT.RT.CRD.IMMOBILERS.548,Crédit rénovation,1251227.0,1380527.0,113593077.0,4.5,CURRENT,Retail,4
3,113772024,2.010055e+09,TN,TN,M,2004-09-30,1969-05-29,PPH,5820,LR,YES,2025-09-05,2029-09-05,Closed,2022-05-27,1011.0,Compte Allocation Touristique TND,TND,2026-01-28,NOT_APPLICABLE,0.000,9000,2725.739,ATB.GRP.CUR.ACCT,ACCOUNTS,ATB.CUR.ACCT.ALL.TOURS.CARTE,Compte Allocation Touristique TND,NaN,NaN,0.0,NaN,NOT_APPLICABLE,Retail,4
4,113772022,2.011092e+09,TN,TN,M,2004-09-30,1960-09-23,PPH,5820,LR,YES,2025-09-05,2029-09-05,Closed,2025-07-02,3611.0,DEPOTS A TERME,TND,2025-06-30,NOT_APPLICABLE,0.000,9000,3300.537,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,DEPOTS A TERME,NaN,NaN,NaN,NaN,NOT_APPLICABLE,Retail,4


In [17]:
# ==========================================================
# 4.4 RESTORE REQUIRED DATA TYPES
# ==========================================================

identifier_columns = [
    "CUSTOMER_NO",
    "ACCOUNT_NO"
]

date_columns = [
    "CUST_OPENING_DATE",
    "DATE_OF_BIRTH",
    "LAST_REVIEW_DATE",
    "NEXT__REVIEW_DATE",
    "ACCT_OPENING_DATE",
    "ACCT_CLOSE_DATE"
]

for column in identifier_columns:
    df[column] = df[column].astype("string")

for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        errors="coerce"
    )

print("Identifier types:")
print(df[identifier_columns].dtypes)

print("\nDate types:")
print(df[date_columns].dtypes)

Identifier types:
CUSTOMER_NO    string
ACCOUNT_NO     string
dtype: object

Date types:
CUST_OPENING_DATE    datetime64[us]
DATE_OF_BIRTH        datetime64[us]
LAST_REVIEW_DATE     datetime64[us]
NEXT__REVIEW_DATE    datetime64[us]
ACCT_OPENING_DATE    datetime64[us]
ACCT_CLOSE_DATE      datetime64[us]
dtype: object


In [18]:
# ==========================================================
# 4.5 INPUT VALIDATION
# ==========================================================

required_columns = [
    "CUSTOMER_NO",
    "ACCOUNT_NO",
    "DATE_OF_BIRTH",
    "CUST_OPENING_DATE",
    "NATIONALITY",
    "RESIDENCE",
    "PARTYCLASS",
    "SCORE_KYC",
    "SALARY",
    "ACCOUNT_STATUS",
    "ACCT_BALANCE",
    "PRODUCT",
    "PRODUCT_GROUP",
    "PRODUCT_LINE"
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_required_columns:
    raise ValueError(
        "Required columns are missing: "
        f"{missing_required_columns}"
    )

print("Input validation successful.")
print(
    "Unique customers:",
    f"{df['CUSTOMER_NO'].nunique():,}"
)
print(
    "Unique accounts:",
    f"{df['ACCOUNT_NO'].nunique(dropna=True):,}"
)
print(
    "Exact duplicates:",
    f"{df.duplicated().sum():,}"
)

Input validation successful.
Unique customers: 363,569
Unique accounts: 410,587
Exact duplicates: 0


## 4.6 Customer Base Construction

The cleaned dataset contains one row per customer-account relationship.

Before creating machine learning features, the dataset must be transformed into a customer-level analytical table.

The target granularity is:

**One row = One unique customer**

In [19]:
# ==========================================================
# CUSTOMER GRANULARITY
# ==========================================================

print("=" * 70)
print("CUSTOMER GRANULARITY")
print("=" * 70)

print(f"Rows               : {len(df):,}")
print(f"Unique Customers   : {df['CUSTOMER_NO'].nunique():,}")
print(f"Unique Accounts    : {df['ACCOUNT_NO'].nunique(dropna=True):,}")

avg_accounts = (
    df["ACCOUNT_NO"].notna().sum()
    / df["CUSTOMER_NO"].nunique()
)

print(f"Average Accounts / Customer : {avg_accounts:.2f}")

CUSTOMER GRANULARITY
Rows               : 490,244
Unique Customers   : 363,569
Unique Accounts    : 410,587
Average Accounts / Customer : 1.23


In [20]:
# ==========================================================
# CUSTOMER BASE TABLE
# ==========================================================

customer_base = (

    df

    .sort_values("CUSTOMER_NO")

    .groupby("CUSTOMER_NO", as_index=False)

    .first()

)

print("=" * 70)
print("CUSTOMER BASE")
print("=" * 70)

print(f"Rows : {len(customer_base):,}")
print(f"Columns : {customer_base.shape[1]}")

display(customer_base.head())

CUSTOMER BASE
Rows : 363,569
Columns : 34


,CUSTOMER_NO,ACCOUNT_NO,NATIONALITY,RESIDENCE,MARITAL_STATUS,CUST_OPENING_DATE,DATE_OF_BIRTH,NATURE_CLIENT,BRANCH,SCORE_KYC,COMPLETED_FILE,LAST_REVIEW_DATE,NEXT__REVIEW_DATE,ACCOUNT_STATUS,ACCT_OPENING_DATE,ACCOUNT_CATEGORY,ACCOUNT_TYPE_DESC,CURRENCY,ACCT_CLOSE_DATE,CLOSURE_REASON,ACCT_BALANCE,INDUSTRY,SALARY,PRODUCT_GROUP,PRODUCT_LINE,PRODUCT,ACCOUNTNATURE,STARTDATE,MATURITYDATE,AMOUNT,FIXEDRATE,PRODUCT_STATUS,PARTYCLASS,LOB
0,112872078,2010326177.0,TN,TN,UNKNOWN,NaT,NaT,UNKNOWN,5801,H2,UNKNOWN,2023-09-22,2025-09-21,Active,2023-02-27,8902.0,AVA ATB,TND,NaT,NOT_APPLICABLE,589747.163,1814,612.0,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,AVA ATB,NaN,NaN,NaN,NaN,NOT_APPLICABLE,Corporate,30
1,112872083,2011458620.0,TN,TN,UNKNOWN,NaT,NaT,UNKNOWN,11,LR,UNKNOWN,NaT,NaT,Active,2025-12-31,8014.0,COMPTE BROKER,TND,NaT,NOT_APPLICABLE,0.000,3100,612.0,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,COMPTE BROKER,NaN,NaN,NaN,NaN,NOT_APPLICABLE,Corporate,30
2,113072080,2010914846.0,TN,TN,UNKNOWN,NaT,NaT,PM,5802,MR,UNKNOWN,NaT,NaT,Active,2025-02-03,8930.0,DEPOT A TERME CLIENT NON ATB,TND,NaT,NOT_APPLICABLE,0.000,111,612.0,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,DEPOT A TERME CLIENT NON ATB,NaN,NaN,NaN,NaN,NOT_APPLICABLE,Corporate,13
3,113073073,2001488537.0,TN,TN,C,2020-10-08,2002-02-20,PPH,5890,LR,YES,2024-09-25,2028-09-24,Closed,NaT,NaN,NOT_APPLICABLE,NOT_APPLICABLE,2024-12-04,ATB.REASON.13,NaN,9000,400.0,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NaN,NaN,NaN,NaN,NOT_APPLICABLE,Retail,4
4,113073074,2003283679.0,TN,TN,M,2020-10-08,1977-08-17,PPH,5890,LR,YES,2025-08-06,2029-08-05,Active,2020-10-08,6001.0,Comptes Spéciaux d épargne,TND,NaT,NOT_APPLICABLE,2001.000,9000,600.0,ATB.GRP.EPARGNE.ACCT,ACCOUNTS,ATB.EPARG.ACCT.SPECIAUX,Comptes Spéciaux d épargne,NaN,NaN,2001.0,NaN,NOT_APPLICABLE,Retail,4


In [21]:
# ==========================================================
# VALIDATION
# ==========================================================

duplicates = customer_base["CUSTOMER_NO"].duplicated().sum()

print("Duplicate customers :", duplicates)

assert duplicates == 0

print("Customer base validation successful.")

Duplicate customers : 0
Customer base validation successful.


## 4.7 Customer Demographics Features

This section creates customer demographic variables that will later be used by the churn prediction model.

Created features:

- AGE
- AGE_GROUP
- CUSTOMER_TENURE_YEARS
- IS_TUNISIAN
- IS_RESIDENT
- CUSTOMER_SEGMENT
- MARITAL_STATUS_GROUP

In [22]:
# ==========================================================
# 4.7 CUSTOMER DEMOGRAPHIC FEATURES
# ==========================================================

TODAY = pd.Timestamp("2026-02-19")

# ----------------------------------------------------------
# AGE
# ----------------------------------------------------------

customer_base["AGE"] = (
    (TODAY - customer_base["DATE_OF_BIRTH"])
    .dt.days
    / 365.25
)

customer_base["AGE"] = customer_base["AGE"].round()

# ----------------------------------------------------------
# AGE GROUP
# ----------------------------------------------------------

customer_base["AGE_GROUP"] = pd.cut(

    customer_base["AGE"],

    bins=[0,25,35,45,55,65,120],

    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56-65",
        "65+"
    ]

)

# ----------------------------------------------------------
# CUSTOMER TENURE
# ----------------------------------------------------------

customer_base["CUSTOMER_TENURE_YEARS"] = (

    (TODAY - customer_base["CUST_OPENING_DATE"])

    .dt.days

    /365.25

).round(1)

# ----------------------------------------------------------
# IS TUNISIAN
# ----------------------------------------------------------

customer_base["IS_TUNISIAN"] = (

    customer_base["NATIONALITY"]

    == "TN"

).astype(int)

# ----------------------------------------------------------
# IS RESIDENT
# ----------------------------------------------------------

customer_base["IS_RESIDENT"] = (

    customer_base["RESIDENCE"]

    == "TN"

).astype(int)

# ----------------------------------------------------------
# CUSTOMER SEGMENT
# ----------------------------------------------------------

customer_base["CUSTOMER_SEGMENT"] = customer_base["PARTYCLASS"]

# ----------------------------------------------------------
# MARITAL STATUS GROUP
# ----------------------------------------------------------

customer_base["MARITAL_STATUS_GROUP"] = (

    customer_base["MARITAL_STATUS"]

)

print("Customer demographic features created successfully.")

Customer demographic features created successfully.


In [23]:
# ==========================================================
# FEATURE VALIDATION
# ==========================================================

features = [

    "AGE",

    "AGE_GROUP",

    "CUSTOMER_TENURE_YEARS",

    "IS_TUNISIAN",

    "IS_RESIDENT",

    "CUSTOMER_SEGMENT",

    "MARITAL_STATUS_GROUP"

]

display(

    customer_base[features].head()

)

,AGE,AGE_GROUP,CUSTOMER_TENURE_YEARS,IS_TUNISIAN,IS_RESIDENT,CUSTOMER_SEGMENT,MARITAL_STATUS_GROUP
0,NaN,NaN,NaN,1,1,Corporate,UNKNOWN
1,NaN,NaN,NaN,1,1,Corporate,UNKNOWN
2,NaN,NaN,NaN,1,1,Corporate,UNKNOWN
3,24.0,18-25,5.4,1,1,Retail,C
4,49.0,46-55,5.4,1,1,Retail,M


In [24]:
# ==========================================================
# DEMOGRAPHIC FEATURE SUMMARY
# ==========================================================

print("="*70)
print("CUSTOMER DEMOGRAPHIC FEATURES")
print("="*70)

summary = pd.DataFrame({

    "Feature":features,

    "Missing":customer_base[features].isna().sum(),

    "Unique Values":[customer_base[c].nunique(dropna=True) for c in features]

})

display(summary)

print()

print(customer_base["AGE"].describe())

print()

print(customer_base["AGE_GROUP"].value_counts(dropna=False))

print()

print(customer_base["CUSTOMER_SEGMENT"].value_counts())

print()

print(customer_base["IS_TUNISIAN"].value_counts())

print()

print(customer_base["IS_RESIDENT"].value_counts())

CUSTOMER DEMOGRAPHIC FEATURES


,Feature,Missing,Unique Values
AGE,AGE,22635,126
AGE_GROUP,AGE_GROUP,22671,6
CUSTOMER_TENURE_YEARS,CUSTOMER_TENURE_YEARS,13548,222
IS_TUNISIAN,IS_TUNISIAN,0,2
IS_RESIDENT,IS_RESIDENT,0,2
CUSTOMER_SEGMENT,CUSTOMER_SEGMENT,0,5
MARITAL_STATUS_GROUP,MARITAL_STATUS_GROUP,0,5



count    340934.000000
mean         47.865587
std          16.965293
min           0.000000
25%          35.000000
50%          47.000000
75%          60.000000
max         126.000000
Name: AGE, dtype: float64

AGE_GROUP
36-45    75877
46-55    68603
26-35    60641
56-65    55035
65+      54897
18-25    25845
NaN      22671
Name: count, dtype: int64

CUSTOMER_SEGMENT
Retail             329845
Corporate Small     26223
Elite                3827
Corporate            3673
Autre                   1
Name: count, dtype: int64

IS_TUNISIAN
1    350064
0     13505
Name: count, dtype: int64

IS_RESIDENT
1    337786
0     25783
Name: count, dtype: int64


## 4.8 Account Portfolio Features

This section aggregates account-level information into customer-level features.

The calculations are based on unique bank accounts to prevent repeated product rows from artificially increasing account counts.

The resulting variables describe:

- number of accounts;
- active and closed accounts;
- account diversity;
- account balances;
- account ownership indicators.

In [25]:
# ============================================================
# 4.8.1 BUILD UNIQUE ACCOUNT-LEVEL TABLE
# ============================================================

def resolve_account_status(series):
    """
    Resolve potentially repeated account statuses.

    Priority:
    1. Active
    2. Closed
    3. NOT_APPLICABLE
    """

    values = set(series.dropna().astype(str))

    if "Active" in values:
        return "Active"

    if "Closed" in values:
        return "Closed"

    return "NOT_APPLICABLE"


account_level = (
    df[df["ACCOUNT_NO"].notna()]
    .groupby(
        ["CUSTOMER_NO", "ACCOUNT_NO"],
        as_index=False
    )
    .agg(
        ACCOUNT_STATUS=(
            "ACCOUNT_STATUS",
            resolve_account_status
        ),

        ACCT_BALANCE=(
            "ACCT_BALANCE",
            "first"
        ),

        ACCOUNT_CATEGORY=(
            "ACCOUNT_CATEGORY",
            "first"
        )
    )
)

print("=" * 70)
print("UNIQUE ACCOUNT-LEVEL TABLE")
print("=" * 70)

print(f"Rows            : {len(account_level):,}")
print(
    "Unique accounts :",
    f"{account_level['ACCOUNT_NO'].nunique():,}"
)

display(account_level.head())

UNIQUE ACCOUNT-LEVEL TABLE
Rows            : 410,587
Unique accounts : 410,587


,CUSTOMER_NO,ACCOUNT_NO,ACCOUNT_STATUS,ACCT_BALANCE,ACCOUNT_CATEGORY
0,112872078,2010311097.0,Active,0.000,8900.0
1,112872078,2010311113.0,Active,-40.372,8902.0
2,112872078,2010326080.0,Active,8263.045,8902.0
3,112872078,2010326112.0,Active,0.000,8902.0
4,112872078,2010326123.0,Active,152606.044,8902.0


In [26]:
# ============================================================
# 4.8.2 ACCOUNT PORTFOLIO AGGREGATION
# ============================================================

portfolio = (
    account_level
    .groupby("CUSTOMER_NO")
    .agg(
        NB_ACCOUNTS=(
            "ACCOUNT_NO",
            "nunique"
        ),

        ACTIVE_ACCOUNTS=(
            "ACCOUNT_STATUS",
            lambda x: (x == "Active").sum()
        ),

        CLOSED_ACCOUNTS=(
            "ACCOUNT_STATUS",
            lambda x: (x == "Closed").sum()
        ),

        TOTAL_BALANCE=(
            "ACCT_BALANCE",
            "sum"
        ),

        AVG_BALANCE=(
            "ACCT_BALANCE",
            "mean"
        ),

        MAX_BALANCE=(
            "ACCT_BALANCE",
            "max"
        ),

        MIN_BALANCE=(
            "ACCT_BALANCE",
            "min"
        ),

        ACCOUNT_DIVERSITY=(
            "ACCOUNT_CATEGORY",
            lambda x: x.nunique(dropna=True)
        )
    )
    .reset_index()
)

balance_columns = [
    "TOTAL_BALANCE",
    "AVG_BALANCE",
    "MAX_BALANCE",
    "MIN_BALANCE"
]

portfolio[balance_columns] = (
    portfolio[balance_columns]
    .fillna(0)
)

portfolio["HAS_MULTIPLE_ACCOUNTS"] = (
    portfolio["NB_ACCOUNTS"] > 1
).astype("int8")

portfolio["HAS_ACTIVE_ACCOUNT"] = (
    portfolio["ACTIVE_ACCOUNTS"] > 0
).astype("int8")

portfolio["HAS_CLOSED_ACCOUNT"] = (
    portfolio["CLOSED_ACCOUNTS"] > 0
).astype("int8")

portfolio["ACTIVE_ACCOUNT_RATIO"] = np.where(
    portfolio["NB_ACCOUNTS"] > 0,
    portfolio["ACTIVE_ACCOUNTS"]
    / portfolio["NB_ACCOUNTS"],
    0
)

display(portfolio.head())

,CUSTOMER_NO,NB_ACCOUNTS,ACTIVE_ACCOUNTS,CLOSED_ACCOUNTS,TOTAL_BALANCE,AVG_BALANCE,MAX_BALANCE,MIN_BALANCE,ACCOUNT_DIVERSITY,HAS_MULTIPLE_ACCOUNTS,HAS_ACTIVE_ACCOUNT,HAS_CLOSED_ACCOUNT,ACTIVE_ACCOUNT_RATIO
0,112872078,59,59,0,748225.554,12681.789051,2833259.331,-1641095.767,17,1,1,0,1.0
1,112872083,14,14,0,0.000,0.000000,0.000,0.000,2,1,1,0,1.0
2,113072080,2,2,0,0.000,0.000000,0.000,0.000,1,1,1,0,1.0
3,113073073,1,0,1,0.000,0.000000,0.000,0.000,0,0,0,1,0.0
4,113073074,1,1,0,2001.000,2001.000000,2001.000,2001.000,1,0,1,0,1.0


In [27]:
# ============================================================
# 4.8.3 ACCOUNT PORTFOLIO VALIDATION
# ============================================================

assert (
    portfolio["ACTIVE_ACCOUNTS"]
    <= portfolio["NB_ACCOUNTS"]
).all(), "Active accounts exceed total accounts."

assert (
    portfolio["CLOSED_ACCOUNTS"]
    <= portfolio["NB_ACCOUNTS"]
).all(), "Closed accounts exceed total accounts."

assert (
    portfolio["ACTIVE_ACCOUNTS"]
    + portfolio["CLOSED_ACCOUNTS"]
    <= portfolio["NB_ACCOUNTS"]
).all(), "Account status counts exceed total accounts."

assert portfolio["CUSTOMER_NO"].duplicated().sum() == 0

print("=" * 70)
print("ACCOUNT PORTFOLIO VALIDATION")
print("=" * 70)

print("Validation successful.")
print(
    "Maximum accounts        :",
    portfolio["NB_ACCOUNTS"].max()
)
print(
    "Maximum active accounts :",
    portfolio["ACTIVE_ACCOUNTS"].max()
)
print(
    "Maximum closed accounts :",
    portfolio["CLOSED_ACCOUNTS"].max()
)
print(
    "Customers in portfolio  :",
    f"{len(portfolio):,}"
)

ACCOUNT PORTFOLIO VALIDATION
Validation successful.
Maximum accounts        : 92
Maximum active accounts : 59
Maximum closed accounts : 92
Customers in portfolio  : 319,129


## 4.8.4 Merge Account Portfolio with Customer Base

The account portfolio is merged with the demographic customer base.

Customers without an account number are retained, and their account-related features are initialized to zero.

In [28]:
# ============================================================
# 4.8.4 MERGE PORTFOLIO INTO CUSTOMER BASE
# ============================================================

portfolio_feature_columns = [
    "CUSTOMER_NO",
    "NB_ACCOUNTS",
    "ACTIVE_ACCOUNTS",
    "CLOSED_ACCOUNTS",
    "TOTAL_BALANCE",
    "AVG_BALANCE",
    "MAX_BALANCE",
    "MIN_BALANCE",
    "ACCOUNT_DIVERSITY",
    "HAS_MULTIPLE_ACCOUNTS",
    "HAS_ACTIVE_ACCOUNT",
    "HAS_CLOSED_ACCOUNT",
    "ACTIVE_ACCOUNT_RATIO"
]

customer_features = customer_base.merge(
    portfolio[portfolio_feature_columns],
    on="CUSTOMER_NO",
    how="left",
    validate="one_to_one"
)

account_count_columns = [
    "NB_ACCOUNTS",
    "ACTIVE_ACCOUNTS",
    "CLOSED_ACCOUNTS",
    "ACCOUNT_DIVERSITY"
]

account_flag_columns = [
    "HAS_MULTIPLE_ACCOUNTS",
    "HAS_ACTIVE_ACCOUNT",
    "HAS_CLOSED_ACCOUNT"
]

account_balance_columns = [
    "TOTAL_BALANCE",
    "AVG_BALANCE",
    "MAX_BALANCE",
    "MIN_BALANCE",
    "ACTIVE_ACCOUNT_RATIO"
]

customer_features[account_count_columns] = (
    customer_features[account_count_columns]
    .fillna(0)
    .astype("int64")
)

customer_features[account_flag_columns] = (
    customer_features[account_flag_columns]
    .fillna(0)
    .astype("int8")
)

customer_features[account_balance_columns] = (
    customer_features[account_balance_columns]
    .fillna(0)
)

print("=" * 70)
print("CUSTOMER FEATURES TABLE")
print("=" * 70)

print(
    "Shape:",
    customer_features.shape
)

print(
    "Duplicate customers:",
    customer_features["CUSTOMER_NO"]
    .duplicated()
    .sum()
)

assert len(customer_features) == 363_569
assert (
    customer_features["CUSTOMER_NO"]
    .duplicated()
    .sum()
    == 0
)

print("Portfolio merge successful.")

display(
    customer_features[
        [
            "CUSTOMER_NO",
            "NB_ACCOUNTS",
            "ACTIVE_ACCOUNTS",
            "CLOSED_ACCOUNTS",
            "TOTAL_BALANCE",
            "HAS_ACTIVE_ACCOUNT",
            "HAS_CLOSED_ACCOUNT"
        ]
    ].head()
)

CUSTOMER FEATURES TABLE
Shape: (363569, 53)
Duplicate customers: 0
Portfolio merge successful.


,CUSTOMER_NO,NB_ACCOUNTS,ACTIVE_ACCOUNTS,CLOSED_ACCOUNTS,TOTAL_BALANCE,HAS_ACTIVE_ACCOUNT,HAS_CLOSED_ACCOUNT
0,112872078,59,59,0,748225.554,1,0
1,112872083,14,14,0,0.000,1,0
2,113072080,2,2,0,0.000,1,0
3,113073073,1,0,1,0.000,0,1
4,113073074,1,1,0,2001.000,1,0


## 4.9 Customer-Level Financial Features

Financial features are created at the customer level using aggregated account balances and the customer salary.

These variables describe:

- income level;
- balance position;
- debtor status;
- financial value;
- balance-to-income relationship.

In [29]:
# ============================================================
# 4.9.1 FINANCIAL DATA PREPARATION
# ============================================================

customer_features["SALARY"] = pd.to_numeric(
    customer_features["SALARY"],
    errors="coerce"
).fillna(0)

customer_features["TOTAL_BALANCE"] = pd.to_numeric(
    customer_features["TOTAL_BALANCE"],
    errors="coerce"
).fillna(0)

customer_features["AVG_BALANCE"] = pd.to_numeric(
    customer_features["AVG_BALANCE"],
    errors="coerce"
).fillna(0)

customer_features["MAX_BALANCE"] = pd.to_numeric(
    customer_features["MAX_BALANCE"],
    errors="coerce"
).fillna(0)

customer_features["MIN_BALANCE"] = pd.to_numeric(
    customer_features["MIN_BALANCE"],
    errors="coerce"
).fillna(0)

print("Financial values prepared.")

Financial values prepared.


In [30]:
# ============================================================
# 4.9.2 FINANCIAL BINARY FEATURES
# ============================================================

customer_features["HAS_POSITIVE_SALARY"] = (
    customer_features["SALARY"] > 0
).astype("int8")

customer_features["HAS_POSITIVE_BALANCE"] = (
    customer_features["TOTAL_BALANCE"] > 0
).astype("int8")

customer_features["HAS_NEGATIVE_BALANCE"] = (
    customer_features["MIN_BALANCE"] < 0
).astype("int8")

customer_features["HAS_ZERO_TOTAL_BALANCE"] = (
    customer_features["TOTAL_BALANCE"] == 0
).astype("int8")

customer_features["HIGH_VALUE_CUSTOMER"] = (
    (customer_features["SALARY"] > 5_000)
    |
    (customer_features["TOTAL_BALANCE"] > 100_000)
).astype("int8")

print("Financial binary features created.")

Financial binary features created.


In [31]:
# ============================================================
# 4.9.3 SALARY LEVEL
# ============================================================

customer_features["SALARY_LEVEL"] = pd.cut(
    customer_features["SALARY"],
    bins=[
        -np.inf,
        0,
        1_000,
        3_000,
        10_000,
        np.inf
    ],
    labels=[
        "NO_POSITIVE_SALARY",
        "LOW",
        "MEDIUM",
        "HIGH",
        "VERY_HIGH"
    ],
    include_lowest=True
)

print(
    customer_features["SALARY_LEVEL"]
    .value_counts(dropna=False)
)

SALARY_LEVEL
LOW                   323923
MEDIUM                 31169
HIGH                    7319
VERY_HIGH                959
NO_POSITIVE_SALARY       199
Name: count, dtype: int64


In [32]:
# ============================================================
# 4.9.4 BALANCE LEVEL
# ============================================================

def classify_balance(value):
    if pd.isna(value):
        return "UNKNOWN"

    if value < 0:
        return "NEGATIVE"

    if value == 0:
        return "ZERO"

    if value <= 1_000:
        return "LOW_POSITIVE"

    if value <= 10_000:
        return "MEDIUM"

    if value <= 100_000:
        return "HIGH"

    return "VERY_HIGH"


customer_features["BALANCE_LEVEL"] = (
    customer_features["TOTAL_BALANCE"]
    .apply(classify_balance)
    .astype("string")
)

print(
    customer_features["BALANCE_LEVEL"]
    .value_counts(dropna=False)
)

BALANCE_LEVEL
ZERO            147764
LOW_POSITIVE    104722
MEDIUM           39139
NEGATIVE         35628
HIGH             29924
VERY_HIGH         6392
Name: count, dtype: Int64


In [33]:
# ============================================================
# 4.9.5 BALANCE TO SALARY RATIO
# ============================================================

customer_features["BALANCE_TO_SALARY_RATIO"] = np.where(
    customer_features["SALARY"] > 0,
    customer_features["TOTAL_BALANCE"]
    / customer_features["SALARY"],
    0
)

customer_features["BALANCE_TO_SALARY_RATIO"] = (
    customer_features["BALANCE_TO_SALARY_RATIO"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

print(
    customer_features["BALANCE_TO_SALARY_RATIO"]
    .describe()
)

count    363569.000000
mean          3.476488
std        1075.448436
min     -119350.185049
25%           0.000000
50%           0.000000
75%           0.551070
max      211913.030000
Name: BALANCE_TO_SALARY_RATIO, dtype: float64


In [34]:
# ============================================================
# 4.9.6 FINANCIAL FEATURE VALIDATION
# ============================================================

financial_features = [
    "SALARY",
    "TOTAL_BALANCE",
    "AVG_BALANCE",
    "MAX_BALANCE",
    "MIN_BALANCE",
    "HAS_POSITIVE_SALARY",
    "HAS_POSITIVE_BALANCE",
    "HAS_NEGATIVE_BALANCE",
    "HAS_ZERO_TOTAL_BALANCE",
    "SALARY_LEVEL",
    "BALANCE_LEVEL",
    "BALANCE_TO_SALARY_RATIO",
    "HIGH_VALUE_CUSTOMER"
]

financial_summary = pd.DataFrame({
    "Feature": financial_features,

    "Missing": [
        customer_features[column]
        .isna()
        .sum()
        for column in financial_features
    ],

    "Unique Values": [
        customer_features[column]
        .nunique(dropna=True)
        for column in financial_features
    ]
})

display(financial_summary)

display(
    customer_features[
        ["CUSTOMER_NO"] + financial_features
    ].head()
)

print("=" * 70)
print("FINANCIAL FEATURE SUMMARY")
print("=" * 70)

print(
    "Rows:",
    f"{len(customer_features):,}"
)

print("\nBalance levels:")
print(
    customer_features["BALANCE_LEVEL"]
    .value_counts(dropna=False)
)

print("\nHigh-value customers:")
print(
    customer_features["HIGH_VALUE_CUSTOMER"]
    .value_counts(dropna=False)
)

assert len(customer_features) == 363_569

assert (
    customer_features["CUSTOMER_NO"]
    .duplicated()
    .sum()
    == 0
)

assert not np.isinf(
    customer_features["BALANCE_TO_SALARY_RATIO"]
).any()

print("\nFinancial feature validation successful.")

,Feature,Missing,Unique Values
0,SALARY,0,6011
1,TOTAL_BALANCE,0,164838
2,AVG_BALANCE,0,165566
3,MAX_BALANCE,0,155552
4,MIN_BALANCE,0,151247
5,HAS_POSITIVE_SALARY,0,2
6,HAS_POSITIVE_BALANCE,0,2
7,HAS_NEGATIVE_BALANCE,0,2
8,HAS_ZERO_TOTAL_BALANCE,0,2
9,SALARY_LEVEL,0,5


,CUSTOMER_NO,SALARY,TOTAL_BALANCE,AVG_BALANCE,MAX_BALANCE,MIN_BALANCE,HAS_POSITIVE_SALARY,HAS_POSITIVE_BALANCE,HAS_NEGATIVE_BALANCE,HAS_ZERO_TOTAL_BALANCE,SALARY_LEVEL,BALANCE_LEVEL,BALANCE_TO_SALARY_RATIO,HIGH_VALUE_CUSTOMER
0,112872078,612.0,748225.554,12681.789051,2833259.331,-1641095.767,1,1,1,0,LOW,VERY_HIGH,1222.590775,1
1,112872083,612.0,0.000,0.000000,0.000,0.000,1,0,0,1,LOW,ZERO,0.000000,0
2,113072080,612.0,0.000,0.000000,0.000,0.000,1,0,0,1,LOW,ZERO,0.000000,0
3,113073073,400.0,0.000,0.000000,0.000,0.000,1,0,0,1,LOW,ZERO,0.000000,0
4,113073074,600.0,2001.000,2001.000000,2001.000,2001.000,1,1,0,0,LOW,MEDIUM,3.335000,0


FINANCIAL FEATURE SUMMARY
Rows: 363,569

Balance levels:
BALANCE_LEVEL
ZERO            147764
LOW_POSITIVE    104722
MEDIUM           39139
NEGATIVE         35628
HIGH             29924
VERY_HIGH         6392
Name: count, dtype: Int64

High-value customers:
HIGH_VALUE_CUSTOMER
0    354960
1      8609
Name: count, dtype: int64

Financial feature validation successful.


## 4.10 Product Portfolio Features

This section summarizes the banking products owned by each customer.

The features describe:

- number and diversity of products;
- product line coverage;
- ownership of loans, savings products, deposits and current accounts.

In [35]:
# ============================================================
# 4.10.1 PRODUCT-LEVEL FLAGS
# ============================================================

product_source = df[
    [
        "CUSTOMER_NO",
        "PRODUCT",
        "PRODUCT_GROUP",
        "PRODUCT_LINE",
        "ACCOUNT_TYPE_DESC",
        "ACCOUNTNATURE"
    ]
].copy()

text_columns = [
    "PRODUCT",
    "PRODUCT_GROUP",
    "PRODUCT_LINE",
    "ACCOUNT_TYPE_DESC",
    "ACCOUNTNATURE"
]

for column in text_columns:
    product_source[column] = (
        product_source[column]
        .astype("string")
        .fillna("")
        .str.upper()
        .str.strip()
    )

product_text = (
    product_source[text_columns]
    .fillna("")
    .agg(" ".join, axis=1)
)

product_source["HAS_LOAN_ROW"] = (
    product_text.str.contains(
        r"CREDIT|CRD|LOAN|LENDING|AVANCE",
        regex=True
    )
).astype("int8")

product_source["HAS_SAVINGS_ROW"] = (
    product_text.str.contains(
        r"EPARGNE|SAVING",
        regex=True
    )
).astype("int8")

product_source["HAS_DEPOSIT_ROW"] = (
    product_text.str.contains(
        r"DEPOT|DEPOSIT|PLACEMENT",
        regex=True
    )
).astype("int8")

product_source["HAS_CURRENT_ACCOUNT_ROW"] = (
    product_text.str.contains(
        r"COMPTE COURANT|CURRENT\.ACCT|CUR\.ACCT",
        regex=True
    )
).astype("int8")

print("Product flags created.")

Product flags created.


In [36]:
# ============================================================
# 4.10.2 PRODUCT PORTFOLIO AGGREGATION
# ============================================================

product_features = (
    product_source
    .groupby("CUSTOMER_NO")
    .agg(
        NB_PRODUCTS=(
            "PRODUCT",
            lambda x: x[
                ~x.isin(["", "NOT_APPLICABLE"])
            ].nunique()
        ),

        NB_PRODUCT_GROUPS=(
            "PRODUCT_GROUP",
            lambda x: x[
                ~x.isin(["", "NOT_APPLICABLE"])
            ].nunique()
        ),

        NB_PRODUCT_LINES=(
            "PRODUCT_LINE",
            lambda x: x[
                ~x.isin(["", "NOT_APPLICABLE"])
            ].nunique()
        ),

        HAS_LOAN=("HAS_LOAN_ROW", "max"),
        HAS_SAVINGS=("HAS_SAVINGS_ROW", "max"),
        HAS_DEPOSIT=("HAS_DEPOSIT_ROW", "max"),
        HAS_CURRENT_ACCOUNT=(
            "HAS_CURRENT_ACCOUNT_ROW",
            "max"
        )
    )
    .reset_index()
)

product_features["PRODUCT_DIVERSITY"] = (
    product_features["NB_PRODUCTS"]
    + product_features["NB_PRODUCT_GROUPS"]
    + product_features["NB_PRODUCT_LINES"]
)

display(product_features.head())

,CUSTOMER_NO,NB_PRODUCTS,NB_PRODUCT_GROUPS,NB_PRODUCT_LINES,HAS_LOAN,HAS_SAVINGS,HAS_DEPOSIT,HAS_CURRENT_ACCOUNT,PRODUCT_DIVERSITY
0,112872078,0,0,0,0,0,0,0,0
1,112872083,0,0,0,0,0,0,0,0
2,113072080,0,0,0,0,0,1,0,0
3,113073073,0,0,0,0,0,0,0,0
4,113073074,1,1,1,0,1,0,0,3


In [37]:
# ============================================================
# 4.10.3 MERGE PRODUCT FEATURES
# ============================================================

customer_features = customer_features.merge(
    product_features,
    on="CUSTOMER_NO",
    how="left",
    validate="one_to_one"
)

product_count_columns = [
    "NB_PRODUCTS",
    "NB_PRODUCT_GROUPS",
    "NB_PRODUCT_LINES",
    "PRODUCT_DIVERSITY"
]

product_flag_columns = [
    "HAS_LOAN",
    "HAS_SAVINGS",
    "HAS_DEPOSIT",
    "HAS_CURRENT_ACCOUNT"
]

customer_features[product_count_columns] = (
    customer_features[product_count_columns]
    .fillna(0)
    .astype("int64")
)

customer_features[product_flag_columns] = (
    customer_features[product_flag_columns]
    .fillna(0)
    .astype("int8")
)

assert len(customer_features) == 363_569
assert customer_features["CUSTOMER_NO"].duplicated().sum() == 0

print("Product feature merge successful.")
print("Shape:", customer_features.shape)

Product feature merge successful.
Shape: (363569, 69)


In [38]:
# ============================================================
# 4.10.4 PRODUCT FEATURE VALIDATION
# ============================================================

product_feature_names = [
    "NB_PRODUCTS",
    "NB_PRODUCT_GROUPS",
    "NB_PRODUCT_LINES",
    "PRODUCT_DIVERSITY",
    "HAS_LOAN",
    "HAS_SAVINGS",
    "HAS_DEPOSIT",
    "HAS_CURRENT_ACCOUNT"
]

product_summary = pd.DataFrame({
    "Feature": product_feature_names,
    "Missing": [
        customer_features[column].isna().sum()
        for column in product_feature_names
    ],
    "Unique Values": [
        customer_features[column].nunique()
        for column in product_feature_names
    ]
})

display(product_summary)

print("\nProduct ownership:")
for column in [
    "HAS_LOAN",
    "HAS_SAVINGS",
    "HAS_DEPOSIT",
    "HAS_CURRENT_ACCOUNT"
]:
    print(
        column,
        ":",
        int(customer_features[column].sum())
    )

print("\nProduct feature validation successful.")

,Feature,Missing,Unique Values
0,NB_PRODUCTS,0,11
1,NB_PRODUCT_GROUPS,0,9
2,NB_PRODUCT_LINES,0,5
3,PRODUCT_DIVERSITY,0,18
4,HAS_LOAN,0,2
5,HAS_SAVINGS,0,2
6,HAS_DEPOSIT,0,2
7,HAS_CURRENT_ACCOUNT,0,2



Product ownership:
HAS_LOAN : 23342
HAS_SAVINGS : 171439
HAS_DEPOSIT : 10611
HAS_CURRENT_ACCOUNT : 21319

Product feature validation successful.


## 4.11 Risk and Compliance Features

This section creates customer-level indicators related to KYC risk, file completeness and review monitoring.

The variables describe:

- KYC risk level;
- completed customer file;
- overdue review status;
- time since the last review;
- time remaining before the next review.

In [39]:
# ============================================================
# 4.11.1 RISK AND COMPLIANCE PREPARATION
# ============================================================

REFERENCE_DATE = pd.Timestamp("2026-02-19")

customer_features["LAST_REVIEW_DATE"] = pd.to_datetime(
    customer_features["LAST_REVIEW_DATE"],
    errors="coerce"
)

customer_features["NEXT__REVIEW_DATE"] = pd.to_datetime(
    customer_features["NEXT__REVIEW_DATE"],
    errors="coerce"
)

customer_features["KYC_RISK_LEVEL"] = (
    customer_features["SCORE_KYC"]
    .astype("string")
    .fillna("UNKNOWN")
    .str.upper()
)

customer_features["FILE_COMPLETED_FLAG"] = (
    customer_features["COMPLETED_FILE"]
    .astype("string")
    .str.upper()
    .eq("YES")
).astype("int8")

print("Risk and compliance data prepared.")

Risk and compliance data prepared.


In [40]:
# ============================================================
# 4.11.2 KYC RISK FLAGS
# ============================================================

customer_features["LOW_KYC_RISK"] = (
    customer_features["KYC_RISK_LEVEL"] == "LR"
).astype("int8")

customer_features["MEDIUM_KYC_RISK"] = (
    customer_features["KYC_RISK_LEVEL"] == "MR"
).astype("int8")

customer_features["HIGH_KYC_RISK"] = (
    customer_features["KYC_RISK_LEVEL"]
    .isin(["H1", "H2", "H3"])
).astype("int8")

customer_features["VERY_HIGH_KYC_RISK"] = (
    customer_features["KYC_RISK_LEVEL"]
    .isin(["H2", "H3"])
).astype("int8")

print("KYC risk flags created.")

KYC risk flags created.


In [41]:
# ============================================================
# 4.11.3 REVIEW MONITORING FEATURES
# ============================================================

customer_features["DAYS_SINCE_LAST_REVIEW"] = (
    REFERENCE_DATE
    - customer_features["LAST_REVIEW_DATE"]
).dt.days

customer_features["DAYS_TO_NEXT_REVIEW"] = (
    customer_features["NEXT__REVIEW_DATE"]
    - REFERENCE_DATE
).dt.days

customer_features["REVIEW_OVERDUE"] = (
    customer_features["DAYS_TO_NEXT_REVIEW"] < 0
).astype("int8")

customer_features["HAS_REVIEW_HISTORY"] = (
    customer_features["LAST_REVIEW_DATE"].notna()
).astype("int8")

customer_features["HAS_NEXT_REVIEW_DATE"] = (
    customer_features["NEXT__REVIEW_DATE"].notna()
).astype("int8")

In [42]:
# ============================================================
# 4.11.4 REVIEW STATUS SEGMENT
# ============================================================

def classify_review_status(days):
    if pd.isna(days):
        return "UNKNOWN"

    if days < 0:
        return "OVERDUE"

    if days <= 30:
        return "DUE_WITHIN_30_DAYS"

    if days <= 90:
        return "DUE_WITHIN_90_DAYS"

    if days <= 365:
        return "DUE_WITHIN_ONE_YEAR"

    return "PLANNED_LATER"


customer_features["REVIEW_STATUS"] = (
    customer_features["DAYS_TO_NEXT_REVIEW"]
    .apply(classify_review_status)
    .astype("string")
)

In [43]:
# ============================================================
# 4.11.5 RISK AND COMPLIANCE VALIDATION
# ============================================================

risk_features = [
    "KYC_RISK_LEVEL",
    "FILE_COMPLETED_FLAG",
    "LOW_KYC_RISK",
    "MEDIUM_KYC_RISK",
    "HIGH_KYC_RISK",
    "VERY_HIGH_KYC_RISK",
    "DAYS_SINCE_LAST_REVIEW",
    "DAYS_TO_NEXT_REVIEW",
    "REVIEW_OVERDUE",
    "HAS_REVIEW_HISTORY",
    "HAS_NEXT_REVIEW_DATE",
    "REVIEW_STATUS"
]

risk_summary = pd.DataFrame({
    "Feature": risk_features,
    "Missing": [
        customer_features[column].isna().sum()
        for column in risk_features
    ],
    "Unique Values": [
        customer_features[column].nunique(dropna=True)
        for column in risk_features
    ]
})

display(risk_summary)

print("\nKYC distribution:")
print(
    customer_features["KYC_RISK_LEVEL"]
    .value_counts(dropna=False)
)

print("\nReview status:")
print(
    customer_features["REVIEW_STATUS"]
    .value_counts(dropna=False)
)

print("\nOverdue reviews:")
print(
    customer_features["REVIEW_OVERDUE"]
    .value_counts(dropna=False)
)

assert len(customer_features) == 363_569
assert customer_features["CUSTOMER_NO"].duplicated().sum() == 0

print("\nRisk and compliance feature validation successful.")

,Feature,Missing,Unique Values
0,KYC_RISK_LEVEL,0,6
1,FILE_COMPLETED_FLAG,0,2
2,LOW_KYC_RISK,0,2
3,MEDIUM_KYC_RISK,0,2
4,HIGH_KYC_RISK,0,2
5,VERY_HIGH_KYC_RISK,0,2
6,DAYS_SINCE_LAST_REVIEW,27386,5006
7,DAYS_TO_NEXT_REVIEW,24555,6795
8,REVIEW_OVERDUE,0,2
9,HAS_REVIEW_HISTORY,0,2



KYC distribution:
KYC_RISK_LEVEL
LR         283629
MR          33563
H2          22110
H1          18691
H3           4746
UNKNOWN       830
Name: count, dtype: Int64

Review status:
REVIEW_STATUS
OVERDUE                178569
PLANNED_LATER          115072
DUE_WITHIN_ONE_YEAR     38607
UNKNOWN                 24555
DUE_WITHIN_90_DAYS       4048
DUE_WITHIN_30_DAYS       2718
Name: count, dtype: Int64

Overdue reviews:
REVIEW_OVERDUE
0    185000
1    178569
Name: count, dtype: int64

Risk and compliance feature validation successful.


### Business Interpretation

The reference date used for compliance monitoring is February 19, 2026, corresponding to the dataset observation date.

A total of 178,569 customers have an overdue KYC review. Customers without a next review date are classified as `UNKNOWN` rather than overdue, because their compliance status cannot be determined without additional business information.

Missing review delays are therefore preserved and handled through explicit availability indicators.

## 4.12 Banking Relationship Features

This section summarizes the customer's organizational relationship with the bank.

The features describe:

- main branch;
- main industry;
- main line of business;
- main account currency;
- number of branches used;
- number of currencies used;
- multi-branch and multi-currency relationships.

In [44]:
# ============================================================
# 4.12.1 DOMINANT VALUE FUNCTION
# ============================================================

def dominant_value(series, default="UNKNOWN"):
    """
    Return the most frequent valid value in a customer group.
    """

    valid_values = (
        series
        .dropna()
        .astype("string")
        .str.strip()
    )

    valid_values = valid_values[
        ~valid_values.isin(
            ["", "UNKNOWN", "NOT_APPLICABLE", "<NA>"]
        )
    ]

    if valid_values.empty:
        return default

    return valid_values.mode().iloc[0]

In [46]:
# ============================================================
# 4.12.2 BANKING RELATIONSHIP AGGREGATION - OPTIMIZED
# ============================================================

banking_source = df[
    [
        "CUSTOMER_NO",
        "BRANCH",
        "INDUSTRY",
        "LOB",
        "CURRENCY"
    ]
].copy()

for column in ["BRANCH", "INDUSTRY", "LOB", "CURRENCY"]:
    banking_source[column] = (
        banking_source[column]
        .astype("string")
        .str.strip()
        .replace(
            {
                "": pd.NA,
                "UNKNOWN": pd.NA,
                "NOT_APPLICABLE": pd.NA,
                "<NA>": pd.NA
            }
        )
    )


def fast_mode_by_customer(data, value_column, output_column):
    valid = data[
        ["CUSTOMER_NO", value_column]
    ].dropna()

    counts = (
        valid
        .groupby(
            ["CUSTOMER_NO", value_column],
            observed=True
        )
        .size()
        .reset_index(name="FREQUENCY")
    )

    result = (
        counts
        .sort_values(
            ["CUSTOMER_NO", "FREQUENCY", value_column],
            ascending=[True, False, True]
        )
        .drop_duplicates(
            subset="CUSTOMER_NO",
            keep="first"
        )
        [["CUSTOMER_NO", value_column]]
        .rename(columns={value_column: output_column})
    )

    return result


main_branch = fast_mode_by_customer(
    banking_source,
    "BRANCH",
    "MAIN_BRANCH"
)

main_industry = fast_mode_by_customer(
    banking_source,
    "INDUSTRY",
    "MAIN_INDUSTRY"
)

main_lob = fast_mode_by_customer(
    banking_source,
    "LOB",
    "MAIN_LOB"
)

main_currency = fast_mode_by_customer(
    banking_source,
    "CURRENCY",
    "MAIN_CURRENCY"
)


banking_counts = (
    banking_source
    .groupby("CUSTOMER_NO")
    .agg(
        NB_BRANCHES=("BRANCH", "nunique"),
        NB_INDUSTRIES=("INDUSTRY", "nunique"),
        NB_LOB=("LOB", "nunique"),
        NB_CURRENCIES=("CURRENCY", "nunique")
    )
    .reset_index()
)


banking_features = (
    banking_counts
    .merge(
        main_branch,
        on="CUSTOMER_NO",
        how="left",
        validate="one_to_one"
    )
    .merge(
        main_industry,
        on="CUSTOMER_NO",
        how="left",
        validate="one_to_one"
    )
    .merge(
        main_lob,
        on="CUSTOMER_NO",
        how="left",
        validate="one_to_one"
    )
    .merge(
        main_currency,
        on="CUSTOMER_NO",
        how="left",
        validate="one_to_one"
    )
)


main_columns = [
    "MAIN_BRANCH",
    "MAIN_INDUSTRY",
    "MAIN_LOB",
    "MAIN_CURRENCY"
]

banking_features[main_columns] = (
    banking_features[main_columns]
    .fillna("UNKNOWN")
    .astype("string")
)

banking_features["IS_MULTI_BRANCH"] = (
    banking_features["NB_BRANCHES"] > 1
).astype("int8")

banking_features["IS_MULTI_CURRENCY"] = (
    banking_features["NB_CURRENCIES"] > 1
).astype("int8")


print("=" * 70)
print("BANKING RELATIONSHIP FEATURES")
print("=" * 70)

print("Shape:", banking_features.shape)

display(banking_features.head())

BANKING RELATIONSHIP FEATURES
Shape: (363569, 11)


,CUSTOMER_NO,NB_BRANCHES,NB_INDUSTRIES,NB_LOB,NB_CURRENCIES,MAIN_BRANCH,MAIN_INDUSTRY,MAIN_LOB,MAIN_CURRENCY,IS_MULTI_BRANCH,IS_MULTI_CURRENCY
0,112872078,1,1,1,3,5801,1814,30,TND,0,1
1,112872083,1,1,1,1,11,3100,30,TND,0,0
2,113072080,1,1,1,2,5802,111,13,TDC,0,1
3,113073073,1,1,1,0,5890,9000,4,UNKNOWN,0,0
4,113073074,1,1,1,1,5890,9000,4,TND,0,0


In [47]:
# ============================================================
# 4.12.3 MERGE BANKING FEATURES
# ============================================================

customer_features = customer_features.merge(
    banking_features,
    on="CUSTOMER_NO",
    how="left",
    validate="one_to_one"
)

banking_count_columns = [
    "NB_BRANCHES",
    "NB_INDUSTRIES",
    "NB_LOB",
    "NB_CURRENCIES"
]

banking_flag_columns = [
    "IS_MULTI_BRANCH",
    "IS_MULTI_CURRENCY"
]

banking_main_columns = [
    "MAIN_BRANCH",
    "MAIN_INDUSTRY",
    "MAIN_LOB",
    "MAIN_CURRENCY"
]

customer_features[banking_count_columns] = (
    customer_features[banking_count_columns]
    .fillna(0)
    .astype("int64")
)

customer_features[banking_flag_columns] = (
    customer_features[banking_flag_columns]
    .fillna(0)
    .astype("int8")
)

customer_features[banking_main_columns] = (
    customer_features[banking_main_columns]
    .astype("string")
    .fillna("UNKNOWN")
)

assert len(customer_features) == 363_569
assert customer_features["CUSTOMER_NO"].duplicated().sum() == 0

print("Banking feature merge successful.")
print("Shape:", customer_features.shape)

Banking feature merge successful.
Shape: (363569, 91)


In [48]:
# ============================================================
# 4.13 GLOBAL FEATURE VALIDATION
# ============================================================

print("=" * 70)
print("GLOBAL FEATURE VALIDATION")
print("=" * 70)

print(f"Rows              : {len(customer_features):,}")
print(f"Columns           : {customer_features.shape[1]}")
print(
    "Unique customers  :",
    f"{customer_features['CUSTOMER_NO'].nunique():,}"
)
print(
    "Duplicate customers:",
    customer_features["CUSTOMER_NO"].duplicated().sum()
)

assert len(customer_features) == 363_569
assert customer_features["CUSTOMER_NO"].nunique() == 363_569
assert customer_features["CUSTOMER_NO"].duplicated().sum() == 0

infinite_numeric_values = np.isinf(
    customer_features
    .select_dtypes(include=np.number)
).sum().sum()

print("Infinite numeric values:", int(infinite_numeric_values))

assert infinite_numeric_values == 0

print("\nGlobal feature validation successful.")

GLOBAL FEATURE VALIDATION
Rows              : 363,569
Columns           : 91
Unique customers  : 363,569
Duplicate customers: 0
Infinite numeric values: 0

Global feature validation successful.


In [49]:
# ============================================================
# 4.14 FINAL MISSING VALUE SUMMARY
# ============================================================

final_missing_summary = pd.DataFrame({
    "Column": customer_features.columns,
    "Missing Values": customer_features.isna().sum().values,
    "Missing (%)": (
        customer_features.isna().mean() * 100
    ).round(2).values
})

final_missing_summary = final_missing_summary.sort_values(
    "Missing (%)",
    ascending=False
)

display(final_missing_summary.head(30))

,Column,Missing Values,Missing (%)
28,MATURITYDATE,339258,93.31
30,FIXEDRATE,339259,93.31
27,STARTDATE,339173,93.29
18,ACCT_CLOSE_DATE,246211,67.72
29,AMOUNT,149027,40.99
14,ACCT_OPENING_DATE,138996,38.23
15,ACCOUNT_CATEGORY,138996,38.23
20,ACCT_BALANCE,138996,38.23
1,ACCOUNT_NO,44440,12.22
11,LAST_REVIEW_DATE,27386,7.53


In [50]:
# ============================================================
# 4.15 FEATURE DICTIONARY
# ============================================================

engineered_features = {
    "AGE": "Customer age at the dataset reference date.",
    "AGE_GROUP": "Customer age segment.",
    "CUSTOMER_TENURE_YEARS": "Years since customer relationship opening.",
    "IS_TUNISIAN": "1 if nationality is Tunisian.",
    "IS_RESIDENT": "1 if residence country is Tunisia.",
    "NB_ACCOUNTS": "Number of unique accounts owned by the customer.",
    "ACTIVE_ACCOUNTS": "Number of active unique accounts.",
    "CLOSED_ACCOUNTS": "Number of closed unique accounts.",
    "ACTIVE_ACCOUNT_RATIO": "Share of active accounts among total accounts.",
    "TOTAL_BALANCE": "Sum of account balances.",
    "AVG_BALANCE": "Average account balance.",
    "ACCOUNT_DIVERSITY": "Number of distinct account categories.",
    "HIGH_VALUE_CUSTOMER": "High salary or high total balance indicator.",
    "NB_PRODUCTS": "Number of distinct banking products.",
    "HAS_LOAN": "Customer owns at least one lending product.",
    "HAS_SAVINGS": "Customer owns at least one savings product.",
    "HAS_DEPOSIT": "Customer owns at least one deposit product.",
    "HAS_CURRENT_ACCOUNT": "Customer owns at least one current account.",
    "KYC_RISK_LEVEL": "Customer KYC risk classification.",
    "REVIEW_OVERDUE": "1 if the next KYC review date is overdue.",
    "REVIEW_STATUS": "Segment based on time to next review.",
    "NB_BRANCHES": "Number of branches associated with the customer.",
    "NB_CURRENCIES": "Number of account currencies used.",
    "IS_MULTI_BRANCH": "1 if customer is associated with multiple branches.",
    "IS_MULTI_CURRENCY": "1 if customer uses multiple currencies."
}

feature_dictionary = pd.DataFrame(
    engineered_features.items(),
    columns=["Feature", "Business Definition"]
)

display(feature_dictionary)

,Feature,Business Definition
0,AGE,Customer age at the dataset reference date.
1,AGE_GROUP,Customer age segment.
2,CUSTOMER_TENURE_YEARS,Years since customer relationship opening.
3,IS_TUNISIAN,1 if nationality is Tunisian.
4,IS_RESIDENT,1 if residence country is Tunisia.
5,NB_ACCOUNTS,Number of unique accounts owned by the customer.
6,ACTIVE_ACCOUNTS,Number of active unique accounts.
7,CLOSED_ACCOUNTS,Number of closed unique accounts.
8,ACTIVE_ACCOUNT_RATIO,Share of active accounts among total accounts.
9,TOTAL_BALANCE,Sum of account balances.


In [51]:
# ============================================================
# 4.16 EXPORT CUSTOMER FEATURE DATASET
# ============================================================

FEATURE_DATA_DIR.mkdir(parents=True, exist_ok=True)

customer_features.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

feature_dictionary_file = (
    REPORT_TABLES_DIR
    / "04_feature_dictionary.xlsx"
)

missing_summary_file = (
    REPORT_TABLES_DIR
    / "04_feature_missing_summary.xlsx"
)

feature_dictionary.to_excel(
    feature_dictionary_file,
    index=False
)

final_missing_summary.to_excel(
    missing_summary_file,
    index=False
)

print("=" * 70)
print("FEATURE DATASET EXPORTED")
print("=" * 70)

print(f"Dataset location   : {OUTPUT_FILE.resolve()}")
print(f"Dataset exists     : {OUTPUT_FILE.exists()}")
print(f"Feature dictionary : {feature_dictionary_file.resolve()}")
print(f"Missing summary    : {missing_summary_file.resolve()}")
print(f"Rows               : {len(customer_features):,}")
print(f"Columns            : {customer_features.shape[1]}")

FEATURE DATASET EXPORTED
Dataset location   : C:\Users\Sarra\OneDrive\Desktop\PFE_CHURN_ESB\data\features\customer_feature_dataset.csv
Dataset exists     : True
Feature dictionary : C:\Users\Sarra\OneDrive\Desktop\PFE_CHURN_ESB\outputs\tables\04_feature_dictionary.xlsx
Missing summary    : C:\Users\Sarra\OneDrive\Desktop\PFE_CHURN_ESB\outputs\tables\04_feature_missing_summary.xlsx
Rows               : 363,569
Columns            : 91


In [52]:
print("=" * 70)
print("NOTEBOOK 04 COMPLETED")
print("=" * 70)

print("✓ Customer-level granularity created")
print("✓ Demographic features created")
print("✓ Account portfolio features created")
print("✓ Financial features created")
print("✓ Product features created")
print("✓ Risk and compliance features created")
print("✓ Banking relationship features created")
print("✓ Final dataset validated and exported")

print()
print("Final shape:", customer_features.shape)

NOTEBOOK 04 COMPLETED
✓ Customer-level granularity created
✓ Demographic features created
✓ Account portfolio features created
✓ Financial features created
✓ Product features created
✓ Risk and compliance features created
✓ Banking relationship features created
✓ Final dataset validated and exported

Final shape: (363569, 91)
